# DG-4 walkthrough — failure-envelope via the parity-aware $r_4$ ratio

**Decision Gate:** DG-4 (failure envelope: cause-labelled, reproducible failure regime).
**Verdict status:** PASS at D1 v0.1.2 (2026-05-06) via picture-fixed Path B numerical $L_4$ extraction. The earlier v0.1.1 PASS verdict at tag `v0.5.0` was [superseded on review](../logbook/2026-05-06_dg-4-pass-path-b-superseded.md) the same day for two HIGH-severity defects in the Path B extraction; v0.1.2 consumes the three repair commits (picture transform, operational `omega_max_factor`, audit-complete result JSON).
**Card exercised:** [D1 v0.1.2](../benchmarks/benchmark_cards/D1_failure-envelope-convergence_v0.1.2.yaml).
**Authoritative status:** [`docs/validity_envelope.md`](../docs/validity_envelope.md).

This notebook reproduces the v0.1.2 PASS pipeline on a reduced fixture for pedagogical purposes — the cells run in seconds and show how the Path B route, the runner, and the metric integrate end-to-end. The full frozen run uses `n_bath_modes=4`, `n_levels_per_mode=3`, a 200-point time grid, and a 20-point `coupling_strength` sweep, takes ~37 s on the steward laptop, and reports `verdict = PASS` with `max baseline r_4 ≈ 47.42`. The reduced fixture below uses `n_bath_modes=2`, `n_levels_per_mode=2`, an 11-point time grid, and a 4-point sweep so the cells run in seconds; the upper half of the sweep range still classifies as `convergence_failure`, which is sufficient for PASS.

DG-4 PASS condition (Sail v0.5 §9): identify at least one reproducible, cause-labelled failure regime. The metric is the parity-aware even-order ratio

$$
r_4(\alpha^2) \;=\; \alpha^2\, \frac{\langle \lVert L_4^{\text{dis}}\rVert\rangle_t}{\langle \lVert L_2^{\text{dis}}\rVert\rangle_t},
\qquad L_n^{\text{dis}} := L_n + i\,[K_n,\,\cdot\,].
$$

Under v0.1.2, the runner's pipeline:

1. Reconstructs raw Schrödinger-picture maps from `benchmarks.exact_finite_env.build_spin_boson_sigma_x_thermal_total`.
2. Subtracts the closed-system baseline at $\alpha = 0$ and least-squares fits the even-power expansion $\Lambda_t(\alpha) = \Lambda_0 + \alpha^2 \Lambda_2 + \alpha^4 \Lambda_4 + O(\alpha^6)$.
3. **Transforms $\Lambda_2, \Lambda_4$ to the interaction picture** by conjugating with $\mathrm{Ad}\,U^\dagger(t)$ where $U(t) = e^{-i H_S t}$ — this is the v0.1.2 picture fix that makes the order-4 extractor's $L_2 = \dot\Lambda_2$, $L_4 = \dot\Lambda_4 - L_2 \Lambda_2$ formula valid.
4. Extracts $L_2, L_4$, computes $K_n$ via Letter Eq. (6), forms the dissipator residual $L_n + i[K_n, \cdot]$, and time-averages its Liouville–Frobenius norm.
5. Re-runs the four reproducibility perturbations (`upper_cutoff_factor ∈ {20, 40}` threaded into the finite-env builder's `omega_max_factor`; `omega_c ∈ {9, 11}` via direct model-spec mutation).

A swept point classifies as `convergence_failure` iff $r_4 > 1$ at baseline **and** stays $> 1$ under all four reproducibility perturbations. The PASS predicate is now non-trivial because all four perturbations are operational under v0.1.2.

> **Caveat — Path B vs Path A.** The order-4 generator $L_4$ is reconstructed via the *numerical* Richardson route (Path B). Path B carries a documented finite-environment extraction floor characterised by the 2026-05-06 σ_z thermal zero-oracle pilot (~few × 10⁻² at default truncation). Analytic Path A (Companion Sec. IV closed form for $L_4$) remains preferred for machine-precision cross-validation and pending.


## 0. Setup

In [1]:
import copy

import numpy as np
import yaml

import cbg
from reporting import benchmark_card as bc
from benchmarks import exact_finite_env, numerical_tcl_extraction as nte

print(f"cbg.__version__       = {cbg.__version__}")
print(f"cbg.__sail_version__  = {cbg.__sail_version__}")
print(f"cbg.__ledger_anchor__ = {cbg.__ledger_anchor__}")

cbg.__version__       = 0.3.0.dev0
cbg.__sail_version__  = 0.5
cbg.__ledger_anchor__ = CL-2026-005_v0.4


## 1. Load the frozen card

Card D1 v0.1.1 freezes the σ_x thermal model with the parity-aware ratio metric. We load it via the schema-aware loader; this also validates the card against `SCHEMA.md v0.1.3`.

In [2]:
from pathlib import Path

D1_PATH = Path(cbg.__file__).parent.parent / "benchmarks" / "benchmark_cards" / "D1_failure-envelope-convergence_v0.1.2.yaml"
card = bc.load_card(D1_PATH)
print(f"card_id           = {card.card_id} {card.version}")
print(f"dg_target         = {card.dg_target}")
print(f"model             = {card.model}")
print(f"target_observable = {card.frozen_parameters['comparison']['target_observable']}")
print(f"threshold         = {card.threshold}")
sweep_range = card.frozen_parameters["sweep"]["sweep_range"]
print(f"sweep             = coupling_strength: {sweep_range['start']} → {sweep_range['end']} ({sweep_range['n_points']} pts, {sweep_range['scheme']})")

card_id           = D1 v0.1.2
dg_target         = DG-4
model             = spin_boson_sigma_x
target_observable = r_4 = <||L_4^dissipator||>_t / <||L_2^dissipator||>_t
threshold         = 1.0
sweep             = coupling_strength: 0.05 → 1.0 (20 pts, log_uniform)


## 2. Path B coefficient computation — $\langle\lVert L_2^{\text{dis}}\rVert\rangle_t$ and $\langle\lVert L_4^{\text{dis}}\rVert\rangle_t$

Path B fits the alpha-power expansion
$$
\Lambda_t(\alpha) \;=\; \Lambda_0 \;+\; \alpha^2\,\Lambda_2 \;+\; \alpha^4\,\Lambda_4 \;+\; O(\alpha^6)
$$
to finite-environment process tomography, takes time derivatives to recover $L_2$ and $L_4$, and reports the time-averaged Liouville-Frobenius norms of the dissipator residuals.

These are *coefficients* in the alpha-expansion, so they are independent of the swept `coupling_strength = α²` value and a single fit covers the entire alpha-sweep.

In [3]:
from models import spin_boson_sigma_x

model_spec = copy.deepcopy(card.frozen_parameters["model"])
# v0.1.2 picture fix: H_S is required so the runner can transform raw
# Schrödinger-picture maps to the interaction picture before extracting L_2, L_4.
H_S, _A = spin_boson_sigma_x.system_arrays_from_spec(model_spec)

# Reduced fixture for fast walkthrough: smaller bath, coarser time grid.
t_grid = np.linspace(0.0, 1.0, 11)
alpha_values = (0.01, 0.02, 0.03)

coeffs = nte.path_b_dissipator_norm_coefficients(
    builder=exact_finite_env.build_spin_boson_sigma_x_thermal_total,
    model_spec=model_spec,
    t_grid=t_grid,
    alpha_values=alpha_values,
    builder_kwargs={"n_bath_modes": 2, "n_levels_per_mode": 2},
    system_hamiltonian=H_S,
)

print(f"fit_relative_residual    = {coeffs.fit_relative_residual:.3e}")
print(f"<||L_2^dis||>_t          = {coeffs.l2_avg:.3e}")
print(f"<||L_4^dis||>_t          = {coeffs.l4_avg:.3e}")
print(f"coefficient ratio l4/l2  = {coeffs.l4_avg / coeffs.l2_avg:.3e}")

fit_relative_residual    = 2.026e-07
<||L_2^dis||>_t          = 1.007e+01
<||L_4^dis||>_t          = 5.839e+01
coefficient ratio l4/l2  = 5.797e+00


## 3. Per-α evaluation of the parity-aware $r_4$ ratio

The runner scales the $\ell_4 / \ell_2$ coefficient ratio by $\alpha^2$ at each swept value of `coupling_strength`. The threshold is $1.0$: any $\alpha$ with $r_4 > 1$ is a *failing candidate*.

In [4]:
sweep_alphas_sq = np.geomspace(sweep_range["start"], sweep_range["end"], num=4)
ratio_coeff = coeffs.l4_avg / coeffs.l2_avg
r_4_baseline = sweep_alphas_sq * ratio_coeff

print("alpha²       r_4_baseline     classification")
print("-" * 50)
for a2, r4 in zip(sweep_alphas_sq, r_4_baseline):
    label = "convergence_failure_candidate" if r4 > 1.0 else "passing"
    print(f"{a2:8.4f}    {r4:10.3e}     {label}")

alpha²       r_4_baseline     classification
--------------------------------------------------
  0.0500     2.898e-01     passing
  0.1357     7.868e-01     passing
  0.3684     2.136e+00     convergence_failure_candidate
  1.0000     5.797e+00     convergence_failure_candidate


## 4. Running the full DG-4 verdict via `run_card`

The runner pipeline (`reporting.benchmark_card._run_dg4_sweep`) does what we did manually above, and additionally re-runs Path B under each reproducibility perturbation (`upper_cutoff_factor ∈ {20, 40}` threaded into the finite-env builder's `omega_max_factor`; `omega_c ∈ {9, 11}` via direct model-spec mutation). A failing candidate is reclassified `convergence_failure` iff $r_4 > 1$ stays under all four perturbations; otherwise `truncation_artefact`.

We use a reduced fixture with `n_points_sweep=4`, `t_end=1.0`, `n_t=11`, `n_bath_modes=2`, `n_levels_per_mode=2`. The full frozen card uses `n_points_sweep=20`, `t_end=20.0`, `n_t=200`, `n_bath_modes=4`, `n_levels_per_mode=3`; the verdict on the full card is recorded in [`benchmarks/results/D1_failure-envelope-convergence_v0.1.2_result.json`](../benchmarks/results/D1_failure-envelope-convergence_v0.1.2_result.json) (audit-complete: per-α + per-α-per-perturbation `r_4` persisted) and in the [v0.1.2 verdict logbook entry](../logbook/2026-05-06_dg-4-pass-path-b-v012.md).

In [5]:
data = copy.deepcopy(yaml.safe_load(open(D1_PATH)))
data["frozen_parameters"]["numerical"]["time_grid"]["t_end"] = 1.0
data["frozen_parameters"]["numerical"]["time_grid"]["n_points"] = 11
data["frozen_parameters"]["sweep"]["sweep_range"]["n_points"] = 4
data["frozen_parameters"]["numerical"]["path_b"] = {
    "alpha_values": [0.01, 0.02, 0.03],
    "n_bath_modes": 2,
    "n_levels_per_mode": 2,
}
bc.validate_card_data(data)
reduced_card = bc._data_to_card(data)

result = bc.run_card(reduced_card)
print(f"verdict          = {result.verdict}")
print(f"runner_version   = {result.runner_version}")
print(f"# test_cases     = {len(result.test_case_results)}")
tcr = result.test_case_results[0]
print(f"test_case        = {tcr.name}")
print(f"max baseline r_4 = {tcr.error:.3e}  (threshold = {tcr.threshold})")

verdict          = PASS
runner_version   = 0.1.0
# test_cases     = 1
test_case        = dg4_failure_envelope_sweep
max baseline r_4 = 3.608e+00  (threshold = 1.0)


The runner notes record the per-α classifications, the Path B fit residual, the time-averaged norm coefficients, and the Path B finite-env caveat. The relevant lines are reproduced below.

In [6]:
relevant_lines = [
    line for line in result.notes.splitlines()
    if any(token in line for token in (
        "verdict", "Verdict", "PASS", "FAIL", "CONDITIONAL",
        "convergence_failure", "truncation_artefact",
        "Path B", "finite-env", "alpha_crit",
    ))
]
print("\n".join(relevant_lines[:20]))

L_4 source: Path B numerical Richardson extraction.
Path B params: alpha_values=(0.01, 0.02, 0.03), n_bath_modes=2, n_levels_per_mode=2.
alpha_crit (interpolated, log-linear): 2.4906e-01.
Per-alpha classifications: passing=2, convergence_failure=2.
Path B carries a finite-env extraction floor (sigma_z thermal zero-oracle ~few * 1e-2 at the default truncation per the 2026-05-06 pilot); verdicts are subject to this documented uncertainty until the analytic Path A (Companion Sec. IV) lands.


## 5. Outcome

On the reduced fixture, the upper two of four swept `coupling_strength` points classify as `convergence_failure`, and the runner interpolates `α_crit` inside the swept range (between the second and third swept points). That alone is sufficient for the DG-4 PASS condition: a reproducible, cause-labelled failure regime exists.

The full frozen run (`n_bath_modes=4`, `n_levels_per_mode=3`, 200-point time grid, 20-point sweep from 0.05 to 1.0) reports a much larger $\ell_4/\ell_2$ coefficient: every one of the 20 swept points classifies as `convergence_failure` after all four reproducibility perturbations, with maximum baseline `r_4 ≈ 47.42` and minimum perturbed coefficient ratio `≈ 41.47` at `omega_c = 11.0`. `α_crit` is pushed below the lowest swept point. Cause label: `convergence failure` per the §4 cause-label taxonomy in [`docs/benchmark_protocol.md`](../docs/benchmark_protocol.md).

`result.dg4_sweep_data` carries the audit-complete in-memory payload (per-α `r_4_baseline`, per-α-per-perturbation `r_4`, per-perturbation Path B fit residuals + dissipator-norm coefficients), and `reporting.benchmark_card.write_dg4_result_json` serialises it to disk. The frozen result is at [`benchmarks/results/D1_failure-envelope-convergence_v0.1.2_result.json`](../benchmarks/results/D1_failure-envelope-convergence_v0.1.2_result.json).

**v0.1.1 supersedure context.** An earlier v0.1.1 PASS verdict at tag `v0.5.0` (same day) was downgraded on review for two HIGH-severity defects in the Path B extraction — the order-4 formula was applied to raw Schrödinger-picture maps without the `Λ_0⁻¹` similarity / `L_0` correction (wrong picture for `H_S ≠ 0`), and the `upper_cutoff_factor` perturbation was a no-op so the PASS predicate was trivially satisfied. v0.1.2 consumes three repair commits ([picture fix](../logbook/2026-05-06_dg-4-pass-path-b-superseded.md), operational `omega_max_factor`, audit-complete result JSON) and the verdict shown above.

> **Citation note.** This verdict authorises citing the repository for the existence of a finite-coupling DG-4 failure-envelope identification under the D1 v0.1.2 frozen scope; it does **not** authorise citing it as analytic order-4 recursion completion or as a convergence-reliability claim. CL-2026-005 v0.4 Entry 2 remains COMPATIBLE *scope-limited* per [`docs/do_not_cite_as.md`](../docs/do_not_cite_as.md).